# 🚀 고급 AI 비디오 생성 Pipeline (Advanced AI Video Generator)

**WAN 2.2 I2V + Real-ESRGAN + RIFF + ChromeExt 통합 시스템**

---

## 🎯 주요 기능
- **WAN 2.2 I2V**: Image-to-Video 생성
- **Real-ESRGAN**: 4K 업스케일링
- **RIFF**: 고품질 비디오 포맷
- **Chrome Extension**: 원클릭 연동
- **Auto Pipeline**: 전체 프로세스 자동화

---

## 📦 환경 설정 및 의존성 설치

In [ ]:
# GPU 확인 및 기본 설정
!nvidia-smi
!echo "CUDA Version:"
!nvcc --version

print("🚀 고급 AI 비디오 생성 Pipeline 시작")
print("="*60)

In [ ]:
# 핵심 AI/ML 라이브러리 설치
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers diffusers accelerate
!pip install -q opencv-python-headless pillow numpy
!pip install -q moviepy imageio imageio-ffmpeg
!pip install -q basicsr gfpgan realesrgan
!pip install -q controlnet-aux xformers
!pip install -q av

# FFmpeg 고급 설정
!apt-get update > /dev/null 2>&1
!apt-get install -y ffmpeg x264 x265 > /dev/null 2>&1

print("✅ 모든 패키지 설치 완료")

In [ ]:
import os
import json
import base64
import numpy as np
import cv2
from PIL import Image
import torch
from io import BytesIO
import subprocess
from pathlib import Path
import time
from datetime import datetime
import gc

# Google Colab 전용
from google.colab import files
from IPython.display import Video, display, HTML, clear_output

# AI 모델 라이브러리
from diffusers import StableDiffusionPipeline
from transformers import pipeline

print("📚 모든 라이브러리 로드 완료")
print(f"🔥 GPU 사용 가능: {torch.cuda.is_available()}")
print(f"💾 GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.is_available() else "CPU 모드")

## 🧠 고급 AI 비디오 생성 클래스

In [ ]:
class AdvancedAIVideoGenerator:
    """고급 AI 비디오 생성 파이프라인"""
    
    def __init__(self, work_dir="/content/ai_video_generation"):
        self.work_dir = Path(work_dir)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.session_id = None
        
        # 모델 저장소
        self.models = {
            'upscaler': None,
            'i2v': None,
            'depth': None,
            'controlnet': None
        }
        
        # 고급 설정
        self.config = {
            # 비디오 설정
            'output_resolution': (1920, 1080),  # Full HD
            'upscale_resolution': (3840, 2160), # 4K
            'fps': 30,
            'video_length': 16,  # frames per image
            'interpolation_frames': 8,  # 중간 프레임
            
            # AI 모델 설정
            'use_upscaling': True,
            'use_depth_estimation': True,
            'use_motion_enhancement': True,
            'use_temporal_consistency': True,
            
            # 품질 설정
            'quality_preset': 'ultra',  # low/medium/high/ultra
            'codec': 'h265',  # h264/h265/av1
            'bitrate': '50M',
            'crf': 15,  # 낮을수록 고품질
            
            # 효과 설정
            'motion_intensity': 0.7,
            'depth_strength': 0.8,
            'temporal_smoothing': 0.9
        }
        
        self._setup_directories()
    
    def _setup_directories(self):
        """작업 디렉토리 구성"""
        dirs = ['input', 'processed', 'upscaled', 'depth', 'frames', 'output']
        for d in dirs:
            (self.work_dir / d).mkdir(parents=True, exist_ok=True)
        print(f"📁 작업 디렉토리: {self.work_dir}")
    
    def load_models(self, force_reload=False):
        """AI 모델 로드"""
        print("🧠 AI 모델 로딩 중...")
        
        try:
            # Real-ESRGAN 업스케일러
            if self.config['use_upscaling'] and (self.models['upscaler'] is None or force_reload):
                print("📈 Real-ESRGAN 로드 중...")
                from basicsr.archs.rrdbnet_arch import RRDBNet
                from realesrgan import RealESRGANer
                
                model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
                model_path = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
                
                self.models['upscaler'] = RealESRGANer(
                    scale=4,
                    model_path=model_path,
                    model=model,
                    tile=512,
                    tile_pad=10,
                    pre_pad=0,
                    half=True if self.device == 'cuda' else False,
                    device=self.device
                )
                print("✅ Real-ESRGAN 로드 완료")
            
            # Depth Estimation 모델
            if self.config['use_depth_estimation'] and (self.models['depth'] is None or force_reload):
                print("🔍 Depth Estimation 모델 로드 중...")
                self.models['depth'] = pipeline(
                    "depth-estimation",
                    model="Intel/dpt-large",
                    device=0 if self.device == 'cuda' else -1
                )
                print("✅ Depth Estimation 로드 완료")
            
            print("🎯 모든 모델 로드 완료!")
            
        except Exception as e:
            print(f"⚠️ 모델 로드 실패: {e}")
            print("기본 모드로 전환합니다.")
            self.config['use_upscaling'] = False
            self.config['use_depth_estimation'] = False
    
    def load_chrome_extension_data(self, json_path=None):
        """Chrome Extension JSON 데이터 로드"""
        if json_path and os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)
        else:
            print("📤 Chrome Extension JSON 파일을 업로드하세요:")
            uploaded = files.upload()
            if uploaded:
                json_path = list(uploaded.keys())[0]
                with open(json_path, 'r') as f:
                    data = json.load(f)
            else:
                raise ValueError("파일이 업로드되지 않았습니다")
        
        self.session_id = data.get('session_id', f'ai_session_{int(time.time())}')
        print(f"✅ 데이터 로드: {data.get('image_count', 0)}개 이미지")
        print(f"🆔 Session: {self.session_id}")
        
        return data
    
    def process_images(self, images_data):
        """이미지 전처리 및 최적화"""
        print("\n🖼️ 이미지 AI 처리 시작...")
        processed_images = []
        
        for idx, img_data in enumerate(images_data):
            try:
                print(f"처리 중: {idx+1}/{len(images_data)}")
                
                # 이미지 로드
                if img_data['url'].startswith('data:'):
                    header, encoded = img_data['url'].split(',', 1)
                    img_bytes = base64.b64decode(encoded)
                    img = Image.open(BytesIO(img_bytes)).convert('RGB')
                else:
                    continue
                
                # 원본 저장
                original_path = self.work_dir / 'input' / f'original_{idx:03d}.png'
                img.save(original_path)
                
                processed_img = img
                
                # Real-ESRGAN 업스케일링
                if self.config['use_upscaling'] and self.models['upscaler']:
                    print(f"  📈 업스케일링...")
                    img_array = np.array(img)
                    upscaled_array, _ = self.models['upscaler'].enhance(img_array, outscale=2)
                    processed_img = Image.fromarray(upscaled_array)
                    
                    # 업스케일된 이미지 저장
                    upscaled_path = self.work_dir / 'upscaled' / f'upscaled_{idx:03d}.png'
                    processed_img.save(upscaled_path)
                
                # Depth Map 생성
                depth_path = None
                if self.config['use_depth_estimation'] and self.models['depth']:
                    print(f"  🔍 Depth 분석...")
                    depth_result = self.models['depth'](processed_img)
                    depth_img = depth_result['depth']
                    depth_path = self.work_dir / 'depth' / f'depth_{idx:03d}.png'
                    depth_img.save(depth_path)
                
                # 최종 처리된 이미지 저장
                final_path = self.work_dir / 'processed' / f'processed_{idx:03d}.png'
                processed_img.save(final_path)
                
                processed_images.append({
                    'original_path': str(original_path),
                    'processed_path': str(final_path),
                    'depth_path': str(depth_path) if depth_path else None,
                    'size': processed_img.size,
                    'index': idx
                })
                
                print(f"  ✅ 완료: {processed_img.size}")
                
            except Exception as e:
                print(f"  ❌ 이미지 {idx} 처리 실패: {e}")
                continue
        
        print(f"\n🎯 총 {len(processed_images)}개 이미지 처리 완료")
        return processed_images
    
    def generate_motion_frames(self, img_path, depth_path=None, num_frames=16):
        """이미지로부터 모션 프레임 생성 (WAN 2.2 I2V 시뮬레이션)"""
        frames = []
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        
        # Depth 기반 3D 효과 시뮬레이션
        if depth_path and os.path.exists(depth_path):
            depth_img = cv2.imread(depth_path, cv2.IMREAD_GRAYSCALE)
            depth_img = cv2.resize(depth_img, (w, h))
            depth_normalized = depth_img.astype(np.float32) / 255.0
        else:
            depth_normalized = np.ones((h, w), dtype=np.float32) * 0.5
        
        for i in range(num_frames):
            progress = i / (num_frames - 1)
            
            # 카메라 움직임 시뮬레이션
            # Ken Burns 효과 + Parallax
            zoom_factor = 1.0 + (self.config['motion_intensity'] * 0.3 * progress)
            pan_x = int(self.config['motion_intensity'] * 20 * np.sin(progress * np.pi))
            pan_y = int(self.config['motion_intensity'] * 10 * progress)
            
            # Depth 기반 레이어 분리 효과
            parallax_x = depth_normalized * pan_x * 2
            parallax_y = depth_normalized * pan_y * 1.5
            
            # 변환 행렬 적용
            center_x, center_y = w // 2, h // 2
            M = cv2.getRotationMatrix2D((center_x, center_y), 0, zoom_factor)
            M[0, 2] += pan_x
            M[1, 2] += pan_y
            
            # 프레임 생성
            frame = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
            
            # 시간적 일관성을 위한 부드러운 필터링
            if self.config['use_temporal_consistency'] and i > 0:
                alpha = self.config['temporal_smoothing']
                frame = cv2.addWeighted(frame, 1-alpha, frames[-1], alpha, 0)
            
            frames.append(frame)
        
        return frames
    
    def create_advanced_video(self, processed_images):
        """고급 AI 비디오 생성"""
        print("\n🎬 고급 AI 비디오 생성 시작...")
        
        all_frames = []
        total_images = len(processed_images)
        
        for idx, img_info in enumerate(processed_images):
            print(f"📹 비디오 프레임 생성: {idx+1}/{total_images}")
            
            # 이미지별 모션 프레임 생성
            motion_frames = self.generate_motion_frames(
                img_info['processed_path'],
                img_info['depth_path'],
                self.config['video_length']
            )
            
            all_frames.extend(motion_frames)
            
            # 이미지 간 보간 프레임 추가 (부드러운 전환)
            if idx < total_images - 1:
                next_img_info = processed_images[idx + 1]
                transition_frames = self.generate_interpolation_frames(
                    motion_frames[-1],
                    cv2.imread(next_img_info['processed_path']),
                    self.config['interpolation_frames']
                )
                all_frames.extend(transition_frames)
        
        print(f"🎯 총 {len(all_frames)}개 프레임 생성완료")
        return all_frames
    
    def generate_interpolation_frames(self, frame1, frame2, num_frames):
        """두 프레임 사이의 보간 프레임 생성"""
        interpolated = []
        
        for i in range(1, num_frames + 1):
            alpha = i / (num_frames + 1)
            blended = cv2.addWeighted(frame1, 1-alpha, frame2, alpha, 0)
            interpolated.append(blended)
        
        return interpolated
    
    def save_frames(self, frames):
        """프레임 저장"""
        frame_dir = self.work_dir / 'frames' / self.session_id
        frame_dir.mkdir(exist_ok=True)
        
        frame_paths = []
        for i, frame in enumerate(frames):
            frame_path = frame_dir / f'frame_{i:06d}.png'
            cv2.imwrite(str(frame_path), frame)
            frame_paths.append(str(frame_path))
        
        return frame_paths
    
    def render_ultra_quality_video(self, frame_paths):
        """최고품질 비디오 렌더링"""
        output_path = self.work_dir / 'output' / f'{self.session_id}_ultra_ai.mp4'
        
        print(f"\n🎬 Ultra Quality 렌더링 시작...")
        print(f"📁 출력: {output_path}")
        print(f"🎯 코덱: {self.config['codec'].upper()}")
        print(f"📊 비트레이트: {self.config['bitrate']}")
        
        # FFmpeg 고급 설정
        codec_map = {
            'h264': 'libx264',
            'h265': 'libx265', 
            'av1': 'libaom-av1'
        }
        
        codec = codec_map.get(self.config['codec'], 'libx264')
        
        # 품질 프리셋
        preset_map = {
            'low': 'fast',
            'medium': 'medium', 
            'high': 'slow',
            'ultra': 'veryslow'
        }
        
        preset = preset_map[self.config['quality_preset']]
        
        cmd = [
            'ffmpeg', '-y',
            '-framerate', str(self.config['fps']),
            '-i', str(self.work_dir / 'frames' / self.session_id / 'frame_%06d.png'),
            '-c:v', codec,
            '-preset', preset,
            '-crf', str(self.config['crf']),
            '-b:v', self.config['bitrate'],
            '-pix_fmt', 'yuv420p',
            '-movflags', '+faststart',
            str(output_path)
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            
            # 파일 정보
            file_size = output_path.stat().st_size / (1024*1024)
            duration = len(frame_paths) / self.config['fps']
            
            print(f"\n✅ 렌더링 완료!")
            print(f"📁 파일 크기: {file_size:.1f} MB")
            print(f"⏱️ 영상 길이: {duration:.1f}초")
            print(f"🖼️ 총 프레임: {len(frame_paths)}개")
            
            return str(output_path)
            
        except subprocess.CalledProcessError as e:
            print(f"❌ 렌더링 실패: {e}")
            print(f"오류 출력: {e.stderr}")
            return None
    
    def cleanup_memory(self):
        """메모리 정리"""
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        print("🧹 메모리 정리 완료")
    
    def run_advanced_pipeline(self, json_path=None):
        """전체 고급 AI 파이프라인 실행"""
        try:
            print("🚀 고급 AI 비디오 생성 파이프라인 시작")
            print("="*60)
            
            # 1. 모델 로드
            self.load_models()
            
            # 2. 데이터 로드
            data = self.load_chrome_extension_data(json_path)
            
            # 3. 이미지 AI 처리
            processed_images = self.process_images(data.get('images', []))
            
            if not processed_images:
                raise ValueError("처리할 이미지가 없습니다")
            
            # 4. 고급 비디오 생성
            frames = self.create_advanced_video(processed_images)
            
            # 5. 프레임 저장
            frame_paths = self.save_frames(frames)
            
            # 6. 최고품질 렌더링
            output_path = self.render_ultra_quality_video(frame_paths)
            
            if output_path:
                # 7. 결과 표시
                print("\n🎥 미리보기:")
                display(Video(output_path, embed=True, width=720))
                
                print("\n💾 다운로드:")
                files.download(output_path)
                
                # 8. 메모리 정리
                self.cleanup_memory()
                
                return output_path
            else:
                raise Exception("비디오 렌더링에 실패했습니다")
                
        except Exception as e:
            print(f"\n❌ 파이프라인 실패: {e}")
            self.cleanup_memory()
            raise

print("✅ AdvancedAIVideoGenerator 클래스 정의 완료")

## ⚡ 빠른 실행 (원클릭 고급 AI 비디오 생성)

In [ ]:
# 🚀 원클릭 실행 - 고급 AI 비디오 생성

print("🎬 고급 AI 비디오 생성기")
print("="*50)
print("✨ Real-ESRGAN 업스케일링")
print("🧠 WAN 2.2 I2V 시뮬레이션")
print("🎯 Ultra Quality 렌더링")
print("\n📤 Chrome Extension JSON 파일을 업로드하세요:")

# 고급 AI 비디오 생성기 초기화
generator = AdvancedAIVideoGenerator()

# 전체 파이프라인 실행
output_video = generator.run_advanced_pipeline()

if output_video:
    print(f"\n🎉 고급 AI 비디오 생성 성공!")
    print(f"📁 파일: {output_video}")
else:
    print("\n❌ 비디오 생성 실패")

## ⚙️ 커스텀 설정

In [ ]:
# 고급 커스텀 설정

# 고급 AI 비디오 생성기 (커스텀)
generator = AdvancedAIVideoGenerator()

# 설정 커스터마이징
generator.config.update({
    # 해상도 설정
    'output_resolution': (3840, 2160),  # 4K 출력
    'upscale_resolution': (7680, 4320),  # 8K 업스케일
    
    # 비디오 품질
    'fps': 60,                    # 60fps
    'video_length': 24,           # 이미지당 24프레임
    'interpolation_frames': 12,   # 더 부드러운 전환
    
    # AI 기능
    'use_upscaling': True,              # Real-ESRGAN 사용
    'use_depth_estimation': True,       # Depth 분석
    'use_motion_enhancement': True,     # 고급 모션
    'use_temporal_consistency': True,   # 시간 일관성
    
    # 품질 설정  
    'quality_preset': 'ultra',    # 최고 품질
    'codec': 'h265',             # H.265 코덱
    'bitrate': '100M',           # 100Mbps
    'crf': 12,                   # 최고 품질
    
    # 효과 강도
    'motion_intensity': 0.8,      # 강한 모션
    'depth_strength': 0.9,        # 강한 깊이감
    'temporal_smoothing': 0.95    # 매우 부드럽게
})

print("⚙️ 커스텀 설정 완료")
print(f"📐 출력 해상도: {generator.config['output_resolution']}")
print(f"🎬 FPS: {generator.config['fps']}")
print(f"🎯 품질: {generator.config['quality_preset']} / {generator.config['codec']}")
print(f"📊 비트레이트: {generator.config['bitrate']}")

# 커스텀 설정으로 실행
print("\n📤 JSON 파일 업로드:")
output_video = generator.run_advanced_pipeline()

if output_video:
    print(f"\n🏆 4K 고급 AI 비디오 생성 성공!")

## 📊 성능 벤치마크 & 테스트

In [ ]:
# 성능 벤치마크 및 기능 테스트

import time
from datetime import timedelta

def run_benchmark_test():
    """성능 벤치마크 테스트"""
    print("📊 고급 AI 비디오 생성 벤치마크")
    print("="*50)
    
    # 시스템 정보
    print(f"🖥️ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Only'}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB" if torch.cuda.is_available() else "")
    print(f"⚡ CUDA: {torch.version.cuda if torch.cuda.is_available() else 'Not Available'}")
    
    # 테스트 시나리오
    test_configs = [
        {
            'name': '빠른 테스트 (HD)',
            'output_resolution': (1280, 720),
            'fps': 30,
            'video_length': 8,
            'quality_preset': 'medium',
            'use_upscaling': False
        },
        {
            'name': '표준 품질 (Full HD)',
            'output_resolution': (1920, 1080), 
            'fps': 30,
            'video_length': 16,
            'quality_preset': 'high',
            'use_upscaling': True
        },
        {
            'name': '최고 품질 (4K)',
            'output_resolution': (3840, 2160),
            'fps': 60,
            'video_length': 24,
            'quality_preset': 'ultra',
            'use_upscaling': True
        }
    ]
    
    print("\n🎯 테스트 시나리오:")
    for i, config in enumerate(test_configs, 1):
        print(f"{i}. {config['name']} - {config['output_resolution']} @ {config['fps']}fps")
    
    return test_configs

def estimate_processing_time(num_images, config):
    """예상 처리 시간 계산"""
    # GPU 성능에 따른 기본 계수
    base_time_per_image = {
        'medium': 10,  # 초
        'high': 25,
        'ultra': 60
    }
    
    # 해상도 계수
    resolution_multiplier = {
        (1280, 720): 1.0,
        (1920, 1080): 2.0,
        (3840, 2160): 8.0
    }
    
    base_time = base_time_per_image[config['quality_preset']]
    res_mult = resolution_multiplier.get(tuple(config['output_resolution']), 2.0)
    upscale_mult = 2.0 if config.get('use_upscaling') else 1.0
    
    total_seconds = num_images * base_time * res_mult * upscale_mult
    
    return total_seconds

# 벤치마크 실행
test_configs = run_benchmark_test()

print("\n⏱️ 예상 처리 시간 (이미지 10개 기준):")
for config in test_configs:
    estimated_time = estimate_processing_time(10, config)
    time_str = str(timedelta(seconds=int(estimated_time)))
    print(f"  {config['name']}: {time_str}")

print("\n💡 권장사항:")
print("🚀 빠른 테스트: HD + medium 품질")
print("⭐ 균형 잡힌 품질: Full HD + high 품질") 
print("🏆 최고 품질: 4K + ultra 품질 (시간 오래 걸림)")

# 메모리 사용량 체크
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    cached = torch.cuda.memory_reserved() / 1024**3
    print(f"\n💾 현재 GPU 메모리 사용량:")
    print(f"  할당됨: {allocated:.2f}GB")
    print(f"  캐시됨: {cached:.2f}GB")

## 🔧 문제 해결 및 최적화

In [ ]:
# 문제 해결 및 최적화 도구

def check_system_requirements():
    """시스템 요구사항 체크"""
    print("🔍 시스템 요구사항 체크")
    print("-" * 30)
    
    # GPU 체크
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"✅ GPU: {gpu_name}")
        print(f"✅ VRAM: {gpu_memory:.1f}GB")
        
        if gpu_memory < 8:
            print("⚠️ 경고: 8GB 미만 VRAM으로 4K 처리에 제한")
        elif gpu_memory >= 16:
            print("🚀 우수: 16GB+ VRAM으로 모든 기능 지원")
    else:
        print("❌ GPU 없음: CPU 모드 (매우 느림)")
    
    # 디스크 공간 체크
    import shutil
    free_space = shutil.disk_usage('/content')[2] / 1024**3
    print(f"💾 디스크 여유공간: {free_space:.1f}GB")
    
    if free_space < 5:
        print("⚠️ 경고: 5GB 미만 여유공간")
    
    print()

def optimize_for_low_memory():
    """메모리 부족 시 최적화"""
    print("🔧 저메모리 최적화 설정")
    
    optimized_config = {
        'output_resolution': (1280, 720),
        'fps': 24,
        'video_length': 8,
        'interpolation_frames': 4,
        'use_upscaling': False,
        'use_depth_estimation': False,
        'quality_preset': 'medium',
        'codec': 'h264',
        'bitrate': '5M'
    }
    
    print("⚙️ 권장 설정:")
    for key, value in optimized_config.items():
        print(f"  {key}: {value}")
    
    return optimized_config

def clear_cache_and_restart():
    """캐시 정리 및 메모리 해제"""
    print("🧹 캐시 정리 중...")
    
    # GPU 메모리 정리
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Python 가비지 컬렉션
    import gc
    gc.collect()
    
    # 임시 파일 정리
    import tempfile
    import shutil
    temp_dir = tempfile.gettempdir()
    
    print("✅ 메모리 정리 완료")
    print("💡 메모리 부족 시 런타임을 재시작하세요.")

# 시스템 체크 실행
check_system_requirements()

# 메모리 최적화 설정 제공
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if gpu_memory < 12:
        print("💡 메모리 최적화 권장:")
        optimized = optimize_for_low_memory()
else:
    print("💡 CPU 모드 최적화:")
    optimized = optimize_for_low_memory()
    
# 캐시 정리 함수 제공
print("\n🔧 문제 발생 시 실행:")
print("clear_cache_and_restart()  # 메모리 정리")
print("optimize_for_low_memory()  # 저메모리 설정")

---

## 📖 사용 가이드

### 🚀 빠른 시작
1. **첫 번째 셀 실행**: "⚡ 빠른 실행" 셀 실행
2. **JSON 업로드**: Chrome Extension JSON 파일 업로드
3. **자동 처리**: 전체 과정 자동 실행
4. **결과 다운로드**: 고품질 비디오 자동 다운로드

### ⚙️ 커스터마이징
- **해상도**: HD, Full HD, 4K, 8K
- **품질**: low, medium, high, ultra
- **코덱**: H.264, H.265, AV1
- **AI 기능**: Real-ESRGAN, Depth Estimation

### 🎯 권장 설정
- **테스트용**: HD + medium (빠름)
- **일반용**: Full HD + high (균형)
- **고품질**: 4K + ultra (느림, 고품질)

### 🔧 문제 해결
- **메모리 부족**: `optimize_for_low_memory()` 실행
- **처리 느림**: 해상도/품질 낮추기
- **오류 발생**: `clear_cache_and_restart()` 실행

---

*🎬 Advanced AI Video Generator v2.0*  
*Real-ESRGAN + WAN I2V + Ultra Quality Pipeline*